# Goal in life (part 3)

- same nn as cool_heuristic, but labels = stockfish evals instead of game outcomes
- dataset: Lichess/chess-position-evaluations (fen, cp, mate, depth)
- yes this is distillation, its mostly here as an upper bound for what our little mlp can learn

#Challenge 1:
- each fen shows up multiple times (multipv lines + diff depths)
- first row of a fen at a given depth is the best line, so we keep that, deepest depth wins
- cp is white pov already

In [ ]:
# load up everything

import chess
import numpy as np
import pandas as pd
import torch
import random
from datasets import load_dataset

num_positions = 600_000
raw_eval_dataset = load_dataset("Lichess/chess-position-evaluations", split="train", streaming=True)

best = {}   # fen -> (depth, cp, mate)
for row in raw_eval_dataset:
    fen = row['fen']
    if fen not in best:
        if len(best) >= num_positions: break
        best[fen] = (row['depth'], row['cp'], row['mate'])
    elif row['depth'] > best[fen][0]:
        best[fen] = (row['depth'], row['cp'], row['mate'])

evals_df = pd.DataFrame([(fen, d, cp, mate) for fen, (d, cp, mate) in best.items()],
                        columns=['fen', 'depth', 'cp', 'mate'])
print(len(evals_df))
evals_df.head()

In [ ]:
# cp -> tanh(cp / 400) so +4 pawns ~ 0.76, mate -> +-1

def eval_to_label(cp, mate):
    if pd.notna(mate): return 1.0 if mate > 0 else -1.0
    return float(np.tanh(cp / 400))

evals_df['label'] = [eval_to_label(cp, mate) for cp, mate in zip(evals_df['cp'], evals_df['mate'])]
evals_df['label'].describe()

#Challenge 2:
- 600k x 769 float32 is ~1.8gb, so we store as int8 (one hot is 0/1, turn is +-1) and cast per batch
- same one hot as cool_heuristic

In [ ]:
piece_order = [
    chess.Piece(chess.PAWN,   chess.WHITE), chess.Piece(chess.KNIGHT, chess.WHITE),
    chess.Piece(chess.BISHOP, chess.WHITE), chess.Piece(chess.ROOK,   chess.WHITE),
    chess.Piece(chess.QUEEN,  chess.WHITE), chess.Piece(chess.KING,   chess.WHITE),
    chess.Piece(chess.PAWN,   chess.BLACK), chess.Piece(chess.KNIGHT, chess.BLACK),
    chess.Piece(chess.BISHOP, chess.BLACK), chess.Piece(chess.ROOK,   chess.BLACK),
    chess.Piece(chess.QUEEN,  chess.BLACK), chess.Piece(chess.KING,   chess.BLACK),
]
piece_to_index = {piece: index for index, piece in enumerate(piece_order)}

color_to_turn = {
    chess.WHITE: 1,
    chess.BLACK: -1
}

def board_to_numerical_rep(board, dtype=np.float32):
    state_tensor = np.zeros(769, dtype=dtype)
    for square, piece in board.piece_map().items():
        state_tensor[square * 12 + piece_to_index[piece]] = 1
    state_tensor[768] = color_to_turn[board.turn]
    return state_tensor

X = np.zeros((len(evals_df), 769), dtype=np.int8)
for i, fen in enumerate(evals_df['fen']):
    X[i] = board_to_numerical_rep(chess.Board(fen), np.int8)
Y = evals_df['label'].to_numpy(dtype=np.float32)

# shuffle then split
perm = np.random.default_rng(42).permutation(len(X))
X, Y = X[perm], Y[perm]
cutoff = int(0.8 * len(X))

X_train = torch.tensor(X[:cutoff])
X_val   = torch.tensor(X[cutoff:])
Y_train = torch.tensor(Y[:cutoff])
Y_val   = torch.tensor(Y[cutoff:])
print(X_train.shape, X_val.shape)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from ANN import ANN   # same arch as cool_heuristic, so terminal_game can load it

# same as cool_heuristic train, plus weight decay + int8 -> float cast

def train(model, X_train, Y_train, X_val, Y_val, epochs, batch_size, lr, patience, weight_decay=1e-5):
    global_train_error = []
    global_val_error = []
    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)

    mse = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

    best_val = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for i in range(epochs):
        epoch_train_errors = []
        epoch_val_errors = []

        model.train()
        for x_true, y_true in train_loader:
            optimizer.zero_grad()
            y_hat = model(x_true.float()).squeeze(-1)
            train_loss = mse(y_hat, y_true)
            epoch_train_errors.append(train_loss.item())
            train_loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for x_true, y_true in val_loader:
                y_hat = model(x_true.float()).squeeze(-1)
                epoch_val_errors.append(mse(y_hat, y_true).item())

        train_mean = sum(epoch_train_errors) / len(epoch_train_errors)
        val_mean = sum(epoch_val_errors) / len(epoch_val_errors)
        global_train_error.append(train_mean)
        global_val_error.append(val_mean)

        # early stopping bookkeeping, and we keep the best weights this time
        if val_mean < best_val:
            best_val = val_mean
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"epoch {i+1}/{epochs} | train {train_mean:.4f} | val {val_mean:.4f}")

        if epochs_without_improvement >= patience:
            print(f"early stop: no val improvement for {patience} epochs (best val {best_val:.4f})")
            break

    model.load_state_dict(best_state)
    return global_train_error, global_val_error

In [ ]:
model = ANN()
train_errors, val_errors = train(model, X_train, Y_train, X_val, Y_val,
                                 epochs=30, batch_size=512, lr=1e-4, patience=3)

In [ ]:
print("always predict train mean:", ((Y_val - Y_train.mean()) ** 2).mean().item())
print("best val:", min(val_errors))
torch.save(model.state_dict(), "chess_sf_ann.pt")

#Does it actually play
- same sanity checks as terminal_game
- then play it vs material bot, random openings, swap colours

In [ ]:
model.eval()

def deep_network(board_state, color):
    X = torch.from_numpy(board_to_numerical_rep(board_state)).unsqueeze(0)
    with torch.inference_mode():
        raw_score = model(X).item()
    return raw_score if color == chess.WHITE else -raw_score

tests = {
    "black down a queen": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "start":              chess.STARTING_FEN,
    "white down a queen": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
}
for name, fen in tests.items():
    print(f"{name:20s}", deep_network(chess.Board(fen), chess.WHITE))

In [ ]:
from base import MinMax, function

def play_game(white_eval, black_eval, depth, opening_plies, max_plies=200):
    board = chess.Board()
    for _ in range(opening_plies):
        board.push(random.choice(list(board.legal_moves)))
    engines = {chess.WHITE: MinMax(white_eval, chess.WHITE), chess.BLACK: MinMax(black_eval, chess.BLACK)}
    while not board.is_game_over() and board.ply() < max_plies:
        board.push(engines[board.turn].find_best_move(board.copy(), depth, True))
    return board.result()   # "*" if we hit max plies, count as draw

def match(nn_eval, n_games=20, depth=2, seed=0):
    random.seed(seed)
    score = 0
    wins = draws = losses = 0
    for g in range(n_games):
        opening_plies = random.randint(4, 8)
        nn_white = g % 2 == 0
        result = play_game(nn_eval if nn_white else function, function if nn_white else nn_eval, depth, opening_plies)
        nn_result = {"1-0": 1, "0-1": 0}.get(result, 0.5)
        if not nn_white and result in ("1-0", "0-1"): nn_result = 1 - nn_result
        if nn_result == 1: wins += 1
        elif nn_result == 0: losses += 1
        else: draws += 1
        score += nn_result
        print(f"game {g+1}: nn {'white' if nn_white else 'black'} -> {result}")
    print(f"nn vs material: +{wins} ={draws} -{losses}, score {score / n_games:.0%}")

match(deep_network, n_games=20, depth=2)